<a href="https://colab.research.google.com/github/trefftzc/voronoiWithJulia/blob/main/Exploring_JACC_Threads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Exploring JACC using a CPU with several cores

JACC is a library that makes it possible to write parallel code that will run on either GPUs or microprocessors with several cores, withouth having to change the code.

One selects a backend to decide where the code will be executed, either on a CPU with several cores or on a GPU.

In this notebook, the code runs on a CPU with two cores.

In [1]:
using Pkg

In [2]:
Pkg.add("JACC")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed SIMDTypes ──────────────────────── v0.1.0
   Installed BitTwiddlingConvenienceFunctions ─ v0.1.6
   Installed PolyesterWeave ─────────────────── v0.2.2
   Installed Polyester ──────────────────────── v0.7.19
   Installed ThreadingUtilities ─────────────── v0.5.6
   Installed LayoutPointers ─────────────────── v0.1.17
   Installed ManualMemory ───────────────────── v0.1.8
   Installed CloseOpenIntervals ─────────────── v0.1.13
   Installed StrideArraysCore ───────────────── v0.5.9
   Installed StaticArrayInterface ───────────── v1.10.0
   Installed JACC ───────────────────────────── v1.3.1
    Updating `~/.julia/environments/v1.12/Project.toml`
  [0979c8fe] + JACC v1.3.1
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [62783981] + BitTwiddlingConvenienceFunctions v0.1.6
  [fb6a15b2] + CloseOpenIntervals v0.1.13
  [0979c8fe] + JACC v1.3.1
  [10f19ff3] + LayoutPointers v0.1

In [3]:
import JACC

In [4]:
JACC.@init_backend

[ Info: Threads backend loaded with 2 threads


In [5]:
using JACC
import JACC

# JACC.@init_backend

# Kernel to find the closest seed

function distance(seedX::Int64,seedY::Int64,x::Int64,y::Int64)
    diff_x = seedX - x
    diff_y = seedY - y
    sum_squares = diff_x * diff_x + diff_y * diff_y
    result = sqrt(sum_squares)
    res::Int64 = round(Int64, result)
    return res
end

function closestSeed(x,y,nSeeds,seedsX::Vector{Int64},seedsY::Vector{Int64})
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    return closest_seed
end

function closestSeedDevice(x,y,canvasDevice,nSeeds,seedsX,seedsY)
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    canvasDevice[x,y] = closest_seed
end

JACC.@init_backend
nSeeds = 4
sizeCanvas = 64
canvas = zeros(Core.Int64,(sizeCanvas,sizeCanvas))
seedsX = zeros(Core.Int64,nSeeds)
seedsY = zeros(Core.Int64,nSeeds)
seedsX[1] = 1
seedsX[2] = 64
seedsX[3] = 1
seedsX[4] = 64
seedsY[1] = 1
seedsY[2] = 1
seedsY[3] = 64
seedsY[4] = 64
canvas_dev = JACC.to_device(canvas)
seedsX_dev = JACC.to_device(seedsX)
seedsY_dev = JACC.to_device(seedsY)

# Sequential version
println("Output of the sequential version:\n")
@inbounds begin
  for y = 1:sizeCanvas
    for x = 1:sizeCanvas
        canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
   end
  end
end

# Print the results

for x = 1:sizeCanvas
    for y = 1:sizeCanvas
        print(canvas[x,y] )
    end
    println()
end
# Now the parallel version

JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments

canvas2 = JACC.to_host(canvas_dev)
println("Output of the parallel version:\n")
for x = 1:sizeCanvas
    for y = 1:sizeCanvas
        print(canvas2[x,y] )
    end
    println()
end
println("Done")



Output of the sequential version:



[ Info: Threads backend loaded with 2 threads


1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111111111133333333333333333333333333333333
1111111111111111111111111

Now with a larger canvas: 4096 x 4096

In [7]:
# voronoi_with_jacc_size_4096

using JACC
import JACC

const SIZE_AREA_P = 4096
# JACC.@init_backend

# Kernel to find the closest seed

function distance(seedX::Int64,seedY::Int64,x::Int64,y::Int64)
    diff_x = seedX - x
    diff_y = seedY - y
    sum_squares = diff_x * diff_x + diff_y * diff_y
    result = sqrt(sum_squares)
    res::Int64 = round(Int64, result)
    return res
end

function closestSeed(x,y,nSeeds,seedsX::Vector{Int64},seedsY::Vector{Int64})
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    return closest_seed
end

function closestSeedDevice(x,y,canvasDevice,nSeeds,seedsX,seedsY)
    #::Array(::Core.Int8)seedsX,::Array(::Core.Int8)seedsY)
    closest_seed = -1
    closest_dist = typemax(Int64)
    for seed =1:nSeeds
        dist = distance(seedsX[seed],seedsY[seed],x,y)
        if dist < closest_dist
            closest_dist = dist
            closest_seed = seed
        end
    end
    canvasDevice[x,y] = closest_seed
end

JACC.@init_backend
nSeeds = 4
sizeCanvas = SIZE_AREA_P
canvas = zeros(Core.Int64,(sizeCanvas,sizeCanvas))
seedsX = zeros(Core.Int64,nSeeds)
seedsY = zeros(Core.Int64,nSeeds)
seedsX[1] = 1
seedsX[2] = SIZE_AREA_P
seedsX[3] = 1
seedsX[4] = SIZE_AREA_P
seedsY[1] = 1
seedsY[2] = 1
seedsY[3] = SIZE_AREA_P
seedsY[4] = SIZE_AREA_P
canvas_dev = JACC.to_device(canvas)
seedsX_dev = JACC.to_device(seedsX)
seedsY_dev = JACC.to_device(seedsY)

# Sequential version
@time @sync begin
  @inbounds begin
    for y = 1:sizeCanvas
        for x = 1:sizeCanvas
            canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
        end
    end
  end
end

@time @sync begin
  @inbounds begin
    for y = 1:sizeCanvas
        for x = 1:sizeCanvas
            canvas[x,y] = closestSeed(x,y,nSeeds,seedsX,seedsY)
        end
    end
  end
end

# Now the parallel version
@time @sync begin
    JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments
end
canvas2 = JACC.to_host(canvas_dev)


@time @sync begin
    JACC.@parallel_for range=(sizeCanvas,sizeCanvas) closestSeedDevice(canvas_dev,nSeeds, seedsX_dev, seedsY_dev) # cannot use spaces for keyword arguments
end
@time @sync begin
canvas2 = JACC.to_host(canvas_dev)
end

# Compare the results
outputs_match = true
for y = 1:sizeCanvas
    for x = 1:sizeCanvas
        if canvas[x,y] != canvas2[x,y]
            println("Mismatch at ($x,$y): canvas = $(canvas[x,y]), canvas2 = $(canvas2[x,y])")
            global
            outputs_match = false
        end
    end
end

if outputs_match
    println("Outputs match!")
else
    println("Outputs do not match.")
end



  2.522105 seconds (46.17 M allocations: 960.992 MiB, 15.86% gc time, 0.88% compilation time)
  1.846244 seconds (46.16 M allocations: 960.485 MiB, 5.18% gc time)


[ Info: Threads backend loaded with 2 threads


  0.469688 seconds (241.05 k allocations: 11.759 MiB, 28.18% compilation time: 89% of which was recompilation)
  0.327599 seconds (16 allocations: 688 bytes)
  0.000012 seconds (8 allocations: 336 bytes)
Outputs match!
